In [1]:
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy import signal
import os
from os import listdir
from os.path import isfile, join
import re
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
from datetime import datetime

In [17]:
%matplotlib qt

In [61]:
ts, ys = np.loadtxt(r'D:\PhD Thesis Writing\Data\2024-08-23\C1--trial1-fsmoff-aomoff-250ks--00001.csv',skiprows=5, delimiter=',', usecols=(0,1),unpack = True)

In [65]:
ts2, ys2 = np.loadtxt(r'D:\PhD Thesis Writing\Data\2024-08-23\C1--trial1-fsmoff-aomoff-5ms--00001.csv',skiprows=5, delimiter=',', usecols=(0,1),unpack = True)

In [66]:
ts3, ys3 = np.loadtxt(r'D:\PhD Thesis Writing\Data\2024-08-23\C1--trial1-fsmoff-aomoff-50ms--00001.csv',skiprows=5, delimiter=',', usecols=(0,1),unpack = True)

In [244]:
#Initial vs Signal
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=13) 
fig, ax = plt.subplots()
ax.plot(ts, ys)
#ax.plot(ts2, ys2)
#ax.plot(ts3, ys3)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Voltage [mV]')
#ax.set_xlim(-0.05, 0.05)
#ax.set_ylim([0,np.max(ys)+(0.05*np.max(ys))])
#ax.set_ylim([0,0.007])
ax.set_ylim([-0.001,0.007])
ax.grid(alpha=0.3)
ax.set_title('Raw Voltage over Time plot')
from matplotlib.ticker import StrMethodFormatter
from matplotlib.ticker import FuncFormatter

def times_1000(y, pos):
    return f"{y*1000:g}"   # multiply label by 1000, strip redundant zeros

ax = plt.gca()             # or your existing Axes
ax.yaxis.set_major_formatter(FuncFormatter(times_1000))
plt.savefig('AppendixA1_Raw.pdf', dpi=300)

In [89]:
print(len(ts))

5000002


In [73]:
print(len(ys))

5000002


In [91]:
len(ts2)

25000002

In [105]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

def compute_segment_starts(N, nperseg, overlap):
    """
    Return sample indices where each segment starts, matching SciPy's logic.

    Parameters
    ----------
    N : int
        Total number of samples.
    nperseg : int
        Segment length in samples.
    overlap : float or int
        - If float in [0, 1): treated as a fraction of nperseg (e.g., 0.5 for 50%).
        - If int: treated as the number of overlapping samples (noverlap).

    Returns
    -------
    starts : list[int]
        Start indices for each complete segment. Any trailing
        samples that can't form a full segment are discarded (like SciPy).
    nperseg_used : int
        The actual segment length used (min(N, nperseg)), also like SciPy.
    noverlap : int
        Overlap in samples.
    """
    # SciPy behavior: if nperseg > N, it uses nperseg=N (and warns)
    nperseg_used = min(int(nperseg), int(N))

    if isinstance(overlap, float):
        if not (0 <= overlap < 1):
            raise ValueError("overlap as fraction must be in [0, 1).")
        noverlap = int(round(nperseg_used * overlap))
    else:
        noverlap = int(overlap)

    if not (0 <= noverlap < nperseg_used):
        raise ValueError("noverlap must satisfy 0 <= noverlap < nperseg_used.")

    step = nperseg_used - noverlap
    # Only full segments; tail is dropped (SciPy does the same)
    starts = list(range(0, N - nperseg_used + 1, step))
    return starts, nperseg_used, noverlap


def shade_segments(ax, t, nperseg, overlap, *, top=None, cmap_name="tab20", alpha=0.22,
                   label_indices=True):
    """
    Shade segments on an existing time-domain plot.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes containing your time-series line.
    t : array-like
        Time vector (same length as your data).
    nperseg : int
        Segment length (samples).
    overlap : float or int
        Overlap (fraction in [0,1) or samples).
    top : float or None
        Y-value at which to place segment index labels. If None, uses 85% of current ylim.
    cmap_name : str
        Matplotlib colormap for distinct colors (e.g., 'tab20', 'tab10', 'Set3').
    alpha : float
        Transparency for shaded spans.
    label_indices : bool
        Whether to draw segment index numbers centered in each span.

    Returns
    -------
    starts : list[int]
        Segment start indices (samples).
    info : dict
        {'nperseg_used': ..., 'noverlap': ...}
    """
    N = len(t)
    starts, nperseg_used, noverlap = compute_segment_starts(N, nperseg, overlap)
    cmap = get_cmap(cmap_name)
    y_min, y_max = ax.get_ylim()
    if top is None:
        top = y_min + 0.85 * (y_max - y_min)

    for i, s in enumerate(starts):
        a = t[s]
        b = t[s + nperseg_used - 1] if s + nperseg_used - 1 < N else t[-1]
        color = cmap(i % cmap.N)
        ax.axvspan(a, b, facecolor=color, alpha=alpha, edgecolor="none")
        ax.axvline(a, linestyle="--", linewidth=1.2, alpha=0.7, color=color)
        if label_indices:
            ax.text((a + b) / 2, top, f"{i}", ha="center", va="center", fontsize=12)

    ax.set_xlim(t[0], t[-1])
    return starts, {"nperseg_used": nperseg_used, "noverlap": noverlap}


In [99]:


# --- Example usage ---
if __name__ == "__main__":
    fs = 1000
    T = 3.0
    N = int(fs * T)
    t = np.arange(N) / fs
    x = 0.8*np.sin(2*np.pi*8*t) + 0.4*np.sin(2*np.pi*22*t) + 0.3*np.random.default_rng(2).standard_normal(N)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(t, x, linewidth=1)
    ax.set_title("Segments with 50% overlap")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    shade_segments(ax, t, nperseg=512, overlap=0.5)   # 50% as a fraction
    plt.show()

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(t, x, linewidth=1)
    ax.set_title("Segments with 128-sample overlap")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    shade_segments(ax, t, nperseg=512, overlap=128)   # explicit samples
    plt.show()


C:\Users\Ry\AppData\Local\Temp\ipykernel_17532\2886309026.py:81: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = get_cmap(cmap_name)


In [101]:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(ts, ys, linewidth=1)
    ax.set_title("Segments with 50% overlap")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")
    shade_segments(ax, ts, nperseg=1e6, overlap=0.5)   # 50% as a fraction
    plt.show()

C:\Users\Ry\AppData\Local\Temp\ipykernel_17532\2886309026.py:81: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = get_cmap(cmap_name)


In [109]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from matplotlib.transforms import blended_transform_factory

def compute_segment_starts(N, nperseg, overlap):
    nperseg = int(min(nperseg, N))
    if isinstance(overlap, float):
        if not (0 <= overlap < 1): raise ValueError("overlap fraction must be in [0,1).")
        noverlap = int(round(nperseg * overlap))
    else:
        noverlap = int(overlap)
    if not (0 <= noverlap < nperseg): raise ValueError("0 <= noverlap < nperseg required.")
    step = nperseg - noverlap
    starts = list(range(0, N - nperseg + 1, step))
    return starts, nperseg, noverlap

def draw_segment_arrows(ax, t, nperseg, overlap, *,
                        rows=2,  # number of staggered rows to reduce crowding
                        y_offsets=(1.02, 1.08),  # y positions as axes-fraction
                        arrowstyle="<->", lw=1.5, alpha=0.9, cmap_name="tab20",
                        label=True, fontsize=9):
    """
    Draw arrows above the axes showing each segment extent, without covering data.
    """
    N = len(t)
    starts, nseg, _ = compute_segment_starts(N, nperseg, overlap)
    cmap = get_cmap(cmap_name)
    # x in data coords; y in axes-fraction coords (so arrows sit above the top)
    trans = blended_transform_factory(ax.transData, ax.transAxes)

    for i, s in enumerate(starts):
        a = t[s]
        b = t[min(s + nseg - 1, N - 1)]
        row = i % max(1, rows)
        y = y_offsets[row if row < len(y_offsets) else -1]
        color = cmap(i % cmap.N)

        ax.annotate("",
            xy=(a, y), xytext=(b, y),
            xycoords=trans, textcoords=trans,
            arrowprops=dict(arrowstyle=arrowstyle, lw=lw, alpha=alpha, color=color))

        if label:
            ax.text((a + b) / 2, y + 0.015, f"{i}", ha="center", va="bottom",
                    fontsize=fontsize, transform=ax.transData)

    # pad the top so the arrows are visible
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax + 0.05*(ymax - ymin))
    return starts

# --- demo ---
if __name__ == "__main__":
    fs = 1000; T = 3.0; N = int(fs*T)
    t = np.arange(N)/fs
    rng = np.random.default_rng(2)
    x = 0.8*np.sin(2*np.pi*8*t) + 0.4*np.sin(2*np.pi*22*t) + 0.3*rng.standard_normal(N)

    fig, ax = plt.subplots(figsize=(11,4))
    ax.plot(t, x, lw=1)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
    ax.set_title("Welch segments indicated with arrows (nperseg=512, 50% overlap)")
    draw_segment_arrows(ax, t, nperseg=512, overlap=0.5, rows=2)
    plt.show()


C:\Users\Ry\AppData\Local\Temp\ipykernel_17532\1087096972.py:28: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = get_cmap(cmap_name)


In [116]:
import matplotlib.pyplot as plt

def draw_segment_track(ax, t, nperseg, overlap, height=1.0, cmap_name="tab20"):
    """
    Adds a thin track at the top using bars (like a Gantt chart) for each segment.
    """
    starts, nseg, _ = compute_segment_starts(len(t), nperseg, overlap)
    cmap = plt.get_cmap(cmap_name)
    y0 = ax.get_ylim()[1] + 0.02*(ax.get_ylim()[1]-ax.get_ylim()[0])
    for i, s in enumerate(starts):
        a = t[s]
        b = t[min(s + nseg - 1, len(t) - 1)]
        ax.broken_barh([(a, b - a)], (y0, height), facecolors=cmap(i % cmap.N), alpha=0.7)
        ax.text((a+b)/2, y0 + height*0.55, f"{i}", ha="center", va="center", fontsize=8)
    # extend y-limit to make the track visible
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, y0 + height*1.6)


In [195]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl  # <-- for colormaps API

def compute_segment_starts(N, nperseg, overlap):
    nperseg = int(min(nperseg, N))
    if isinstance(overlap, float):
        if not (0 <= overlap < 1):
            raise ValueError("overlap fraction must be in [0,1).")
        noverlap = int(round(nperseg * overlap))
    else:
        noverlap = int(overlap)
    if not (0 <= noverlap < nperseg):
        raise ValueError("0 <= noverlap < nperseg required.")
    step = nperseg - noverlap
    starts = list(range(0, N - nperseg + 1, step))
    return starts, nperseg, noverlap

def draw_segment_arrows(ax, t, nperseg, overlap, *,
                        rows=2,
                        top_margin=0.08,
                        cmap_name="tab20",
                        arrowstyle="<->",
                        arrow_lw=1.6,
                        arrow_alpha=0.95,
                        vline_alpha=0.9,
                        draw_end_lines=True,
                        label=True, fontsize=13):
    """Arrows inside the plot + color-matched vertical lines."""
    N = len(t)
    starts, nseg, _ = compute_segment_starts(N, nperseg, overlap)

    # ✅ Use the new colormaps API, with a fallback for older Matplotlib
    try:
        cmap = mpl.colormaps.get_cmap(cmap_name)
    except AttributeError:  # Matplotlib < 3.6
        from matplotlib.cm import get_cmap as _legacy_get_cmap
        cmap = _legacy_get_cmap(cmap_name)

    y_min, y_max = ax.get_ylim()
    yr = (y_max - y_min)
    y_base = y_max - top_margin * yr
    row_offsets = np.linspace(0.0, 0.06 * yr, max(1, rows))

    colors = []
    for i, s in enumerate(starts):
        a = t[s]
        b = t[min(s + nseg - 1, N - 1)]
        color = cmap((2*i) % cmap.N)
        colors.append(color)

        ax.axvline(a, color='grey', linestyle="--", linewidth=1.1, alpha=vline_alpha*0.5)
        if draw_end_lines:
            ax.axvline(b, color='grey', linestyle="--", linewidth=1.1, alpha=vline_alpha*0.5)

        row = i % max(1, rows)
        y = y_base - row_offsets[row]

        ax.annotate("", xy=(b, y), xytext=(a, y),
                    arrowprops=dict(arrowstyle=arrowstyle, lw=arrow_lw, color=color, alpha=arrow_alpha))

        if label:
            ax.text((a + b) / 2, y, f"{i}", ha="center", va="bottom",
                    fontsize=fontsize, color=color)

    ax.set_ylim(y_min, y_max)
    return starts, {"nperseg_used": nseg,
                    "noverlap": nseg - (starts[1] - starts[0]) if len(starts) > 1 else 0,
                    "colors": colors}


In [246]:
#Initial vs Signal
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=13) 
fig, ax = plt.subplots()
ax.plot(ts, ys)
#ax.plot(ts2, ys2)
#ax.plot(ts3, ys3)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Voltage [mV]')
ax.set_xlim(-10, 10)
#ax.set_ylim([0,np.max(ys)+(0.05*np.max(ys))])
ax.set_ylim([-0.001,0.007])
#ax.grid(alpha=0.3)
ax.set_title('Segmented V/t plot: overlap 50%, npseg=$10^{6}$')
#plt.legend(loc='lower left')
#plt.savefig('HybridVsFibre.pdf', dpi=300)
#shade_segments(ax, ts, nperseg=1e6, overlap=0.5)   # 50% as a fraction
draw_segment_arrows(ax, ts, nperseg=1e6, overlap=0.5, rows=2)
from matplotlib.ticker import StrMethodFormatter
from matplotlib.ticker import FuncFormatter

def times_1000(y, pos):
    return f"{y*1000:g}"   # multiply label by 1000, strip redundant zeros

ax = plt.gca()             # or your existing Axes
ax.yaxis.set_major_formatter(FuncFormatter(times_1000))
plt.savefig('AppendixA1_Segments.pdf', dpi=300)

In [ ]:
#Initial vs Signal
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=13) 
fig, ax = plt.subplots()
ax.plot(ts, ys)
#ax.plot(ts2, ys2)
#ax.plot(ts3, ys3)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Voltage [mV]')
#ax.set_xlim(-0.05, 0.05)
#ax.set_ylim([0,np.max(ys)+(0.05*np.max(ys))])
#ax.set_ylim([0,0.007])
ax.set_ylim([-0.001,0.007])
ax.grid(alpha=0.3)
ax.set_title('Raw Voltage over Time plot')
from matplotlib.ticker import StrMethodFormatter
from matplotlib.ticker import FuncFormatter

def times_1000(y, pos):
    return f"{y*1000:g}"   # multiply label by 1000, strip redundant zeros

ax = plt.gca()             # or your existing Axes
ax.yaxis.set_major_formatter(FuncFormatter(times_1000))
plt.savefig('AppendixA1_Raw.pdf', dpi=300)

In [252]:
wd=signal.windows.hann(int(1e6))

In [268]:
len(wd)

1000000

In [257]:
plt.plot(np.linspace(0,int(1e6),int(1e6)),wd)

In [564]:
tseg=[]
yseg=[]
for i in range(len(wd)):
    tseg.append(ts[i])
    yseg.append(ys[i])

In [276]:
plt.plot(tseg,yseg)

In [283]:
ysegdetrend=yseg-(np.mean(yseg))

In [285]:
yseghann=ysegdetrend*wd

In [287]:
plt.plot(tseg,yseg)
plt.plot(tseg,ysegdetrend)
plt.plot(tseg,yseghann)

In [319]:
#seg0
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=13) 
fig, ax = plt.subplots()
ax.plot(tseg, yseg)
#ax.plot(ts2, ys2)
#ax.plot(ts3, ys3)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Voltage [mV]')
ax.set_xlim(np.min(tseg), np.max(tseg)+0.01)
#ax.set_ylim([0,np.max(ys)+(0.05*np.max(ys))])
#ax.set_ylim([0,0.007])
#ax.set_ylim([-0.001,0.007])
ax.grid(alpha=0.3)
ax.set_title('V/t plot: Segment 0')
from matplotlib.ticker import StrMethodFormatter
from matplotlib.ticker import FuncFormatter

def times_1000(y, pos):
    return f"{y*1000:g}"   # multiply label by 1000, strip redundant zeros

ax = plt.gca()             # or your existing Axes
ax.yaxis.set_major_formatter(FuncFormatter(times_1000))
plt.savefig('AppendixA1_Seg0.pdf', dpi=300)

In [471]:
#Initial vs Signal
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=13) 
fig, ax = plt.subplots()
ax.plot(tseg, ysegdetrend,label='S0 Detrend')
ax.plot(tseg,yseghann,label='S0 Hann Window')
ax.plot(tseg,wd*(np.max(ysegdetrend)), label='Hann Window')
#ax.plot(ts2, ys2)
#ax.plot(ts3, ys3)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Voltage [mV]')
ax.set_xlim(np.min(tseg), np.max(tseg)+0.01)
#ax.set_ylim([0,np.max(ys)+(0.05*np.max(ys))])
#ax.set_ylim([0,0.007])
#ax.set_ylim([-0.001,0.007])
ax.grid(alpha=0.3)
ax.set_title('V/t plot: Segment 0, detrend and Hann window')
from matplotlib.ticker import StrMethodFormatter
from matplotlib.ticker import FuncFormatter

def times_1000(y, pos):
    return f"{y*1000:g}"   # multiply label by 1000, strip redundant zeros

ax = plt.gca()             # or your existing Axes
ax.yaxis.set_major_formatter(FuncFormatter(times_1000))
plt.legend(loc='lower right')
plt.savefig('AppendixA1_Seg0detrend.pdf', dpi=300)

In [321]:
import scipy.fft

In [431]:
ysegfft=scipy.fft.rfft(yseghann)
d=250000
tsegfft=scipy.fft.rfftfreq(len(tseg),d=d)

In [532]:
tsegpsd=scipy.fft.rfftfreq(len(tseg),d=1/(len(ys)/20))

In [433]:
print(d)

250000


In [453]:
ysegfft

array([-148.270386    +0.j        ,  120.06745722+190.89005374j,
         -9.16966641 -66.30869865j, ...,   -0.28546127  +0.9579467j ,
          0.27248365  -0.62767387j,   -0.66297063  +0.j        ])

In [437]:
#len(ysegfft)
len(tsegfft)

500001

In [457]:
ysegfftmag2=[]
ysegfftmag=[]
for i in range(len(ysegfft)):
    ysegfftmag2.append(np.linalg.norm(ysegfft[i])**2)
    ysegfftmag.append(np.linalg.norm(ysegfft[i]))

In [465]:
plt.loglog(tsegfft,ysegfftmag)

In [467]:
plt.loglog(tsegfft,ysegfftmag2)

In [426]:
plt.plot(np.linspace(0,len(tsegfft),len(tsegfft)),tsegfft)

In [391]:
len(tsegfft)

1000000

In [528]:
segfs,segpx=signal.periodogram(yseghann, len(ys)/20, detrend=False)


In [530]:
plt.loglog(segfs,segpx)

In [481]:
wd2=wd**2

In [483]:
wd2sum=np.sum(wd2)

In [598]:
scale=(len(ys)/20)*wd2sum

In [606]:
ysegPxx=ysegfftmag2/scale

In [608]:
plt.loglog(tsegfft,ysegfftmag2/scale)

In [614]:
#Initial vs Signal
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=13) 
fig, ax = plt.subplots()
ax.loglog(tsegpsd,ysegfftmag,label='|FFT|')
ax.loglog(tsegpsd,ysegfftmag2,label='|FFT$|^{2}$')

#ax.plot(ts2, ys2)
#ax.plot(ts3, ys3)
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('FFT [a.u./Hz]')
ax.set_xlim(np.min(tseg), np.max(tseg)+0.01)
#ax.set_ylim([0,np.max(ys)+(0.05*np.max(ys))])
#ax.set_ylim([0,0.007])
#ax.set_ylim([-0.001,0.007])
ax.grid(alpha=0.3)
ax.set_title('Segment 0, |FFT| & |FFT$|^{2}$')
plt.legend(loc='lower right')
plt.savefig('AppendixA1_Seg0FFT.pdf', dpi=300)

C:\Users\Ry\AppData\Local\Temp\ipykernel_17532\133144702.py:13: UserWarning: Attempt to set non-positive xlim on a log-scaled axis will be ignored.
  ax.set_xlim(np.min(tseg), np.max(tseg)+0.01)


In [524]:
fin_fs,fin_Pxx=signal.welch(ys,len(ys)/20, nperseg=int(1e6))

In [526]:
plt.loglog(fin_fs,fin_Pxx)

In [616]:
#Initial vs Signal
plt.rcParams.update({'font.size': 10})
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=13) 
fig, ax = plt.subplots()
ax.loglog(tsegpsd,ysegPxx,label='PSD, Seg 0')
ax.loglog(fin_fs,fin_Pxx, label='PSD, Average')
#ax.plot(ts2, ys2)
#ax.plot(ts3, ys3)
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('PSD [$V^{2}/$Hz]')
ax.set_xlim(np.min(fin_fs), np.max(fin_fs)+0.01)
#ax.set_ylim([0,np.max(ys)+(0.05*np.max(ys))])
#ax.set_ylim([0,0.007])
#ax.set_ylim([-0.001,0.007])
ax.grid(alpha=0.3)
ax.set_title('Single segment PSD & full Welch')
plt.legend(loc='lower right')
plt.savefig('AppendixA1_Welch.pdf', dpi=300)

C:\Users\Ry\AppData\Local\Temp\ipykernel_17532\1953289572.py:12: UserWarning: Attempt to set non-positive xlim on a log-scaled axis will be ignored.
  ax.set_xlim(np.min(fin_fs), np.max(fin_fs)+0.01)


In [566]:
tseg2=[]
yseg2=[]
for i in range(len(wd)):
    j=int(i+(int(1e6)/2))
    tseg2.append(ts[j])
    yseg2.append(ys[j])

In [568]:
yseg2

[0.0016114821,
 0.0010667558,
 0.00086248339,
 0.00068090794,
 0.00090787726,
 0.00072630181,
 0.00097596805,
 0.001339119,
 0.0012710282,
 0.0013618159,
 0.0010667558,
 0.0011575435,
 0.0010213619,
 0.00090787726,
 0.0011802404,
 0.0012937251,
 0.00099866498,
 0.00088518032,
 0.001339119,
 0.00077169567,
 0.00077169567,
 0.0011802404,
 0.0013845128,
 0.0013845128,
 0.0014526036,
 0.00074899874,
 0.0011348466,
 0.0010894527,
 0.0011121496,
 0.00088518032,
 0.00097596805,
 0.00070360487,
 0.00081708953,
 0.0012710282,
 0.00095327112,
 0.0010213619,
 0.00095327112,
 0.00088518032,
 0.0010894527,
 0.0014299067,
 0.0011802404,
 0.0017249668,
 0.0010440588,
 0.00070360487,
 0.00097596805,
 0.00090787726,
 0.00095327112,
 0.00090787726,
 0.00086248339,
 0.00090787726,
 0.00045393863,
 0.00095327112,
 0.0012483312,
 0.00063551408,
 0.0012483312,
 0.0010667558,
 0.00072630181,
 0.00088518032,
 0.00095327112,
 0.00097596805,
 0.0010213619,
 0.00086248339,
 0.00077169567,
 0.0010894527,
 0.00099

In [570]:
ysegdetrend2=yseg2-(np.mean(yseg2))

In [572]:
yseghann2=ysegdetrend2*wd

In [574]:
segfs2,segpx2=signal.periodogram(yseghann2, len(ys)/20, detrend=False)


In [586]:
plt.loglog(segfs2,segpx2)
plt.loglog(segfs,segpx)
plt.loglog(fin_fs,fin_Pxx, label='PSD, Average')